# The forge: a reference from TRELLIS.2 (docs/forge.md)

Turns an asset's concept views into a textured mesh with TRELLIS.2 (Microsoft; MIT code and
weights, docs/citations.md), for `tools/forge reference <asset> --mesh mesh.glb` to turn into
samples. The mesh is working material and never ships.

Licence note: TRELLIS.2's README names dependencies (nvdiffrast, nvdiffrec) under NVIDIA's Source
Code License, for research or evaluation. Read it before using a reference for a commercial
release (docs/forge.md, "Risks and licensing"); the clean chain (fitting to the concept views
alone) avoids it.

**Not yet run.** Written without a Colab session (it needs a GPU with about 24 GB: an A100 or
L4): follow the repository's README for its install and its pipeline's call where they differ, and
correct this notebook in the change that first runs it.

In [ ]:
!git clone --recursive https://github.com/microsoft/TRELLIS.2 /content/TRELLIS.2
%cd /content/TRELLIS.2
!git rev-parse HEAD
# Install as the README says (its setup script and flags change between versions).

In [ ]:
import json, subprocess
from google.colab import drive
from PIL import Image
drive.mount("/content/drive")
ASSET = "table"
SEED = 7
root = f"/content/drive/MyDrive/wrela-forge/{ASSET}"
views = [Image.open(f"{root}/concept/{v}.png") for v in ("three_quarter", "front", "side", "top")]

In [ ]:
import os
# The README's image-to-3D pipeline, from its pretrained 4B model; check the names against it.
from trellis2.pipelines import Trellis2ImageTo3DPipeline
pipe = Trellis2ImageTo3DPipeline.from_pretrained("microsoft/TRELLIS.2-4B")
pipe.cuda()
result = pipe.run(views[0], seed=SEED)
glb = result.to_glb()  # as the README exports it
os.makedirs(f"{root}/reference", exist_ok=True)
glb.export(f"{root}/reference/mesh.glb")
rev = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
json.dump({"model": "microsoft/TRELLIS.2-4B", "version": rev, "seed": SEED,
           "inputs": "concept/three_quarter.png"}, open(f"{root}/reference/made.json", "w"), indent=2)

Then, here: `tools/forge reference <asset> --mesh .forge/<asset>/reference/mesh.glb --model
microsoft/TRELLIS.2-4B --version <rev> --inputs concept/three_quarter.png --seed <seed>`.